# **Deskripsi Dataset**

Dataset yang digunakan merupakan gambaran tentang mahasiswa yang terdaftar dalam 
berbagai program sarjana yang ditawarkan oleh perguruan tinggi. Data ini mencakup data 
demografis, faktor sosial-ekonomi, dan informasi kinerja akademik yang dapat digunakan 
untuk menganalisis faktor-faktor yang mungkin memprediksi tingkat kesuksesan akademik 
mahasiswa.

# **Requirements and Config**

In [5]:
# !pip install seaborn matplotlib numpy pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class settings:
    DATA_DIR = "../dataset/"
    TRAIN_FILE = "train.csv"
    TEST_FILE = "test.csv"
    SUBMISSION_FILE = "sample_submission.csv"
    SEED = 42
    TARGET = "Target"
    TRAIN_PATH = DATA_DIR + TRAIN_FILE
    TEST_PATH = DATA_DIR + TEST_FILE
    SUBMISSION_PATH = DATA_DIR + SUBMISSION_FILE    

# **Exploratory Data Analysis**

# **Preprocessing**

## **Data Cleaning**

In [ ]:
def drop_columns(df):
    col = []
    return df.drop(columns=col)

## **Data Transformation**

## **Feature Selection**

## **Dimensionality Reduction**

## **Pipeline**

In [ ]:
class Preprocessing:
    def fit(self, X):
        pass
    
    def transform(self, X):
        pass
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# **Modeling and Validation**

## **Metrics**

In [ ]:
from abc import ABC, abstractmethod

class Metric(ABC):
    """
    Metric interface: all metrics must implement __call__
    """

    @abstractmethod
    def __call__(self, y_true, y_pred):
        pass


# ============================================================
# Utility: Confusion Matrix
# ============================================================

class ConfusionMatrix:
    """
    Computes confusion matrix using NumPy.
    """

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        labels = np.unique(np.concatenate([y_true, y_pred]))
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        
        cm = np.zeros((len(labels), len(labels)), dtype=int)

        for t, p in zip(y_true, y_pred):
            cm[label_to_idx[t], label_to_idx[p]] += 1
        
        return cm, labels


# ============================================================
# Accuracy
# ============================================================

class Accuracy(Metric):
    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return float(np.mean(y_true == y_pred))


# ============================================================
# Precision Macro
# ============================================================

class PrecisionMacro(Metric):
    """
    Macro-averaged precision (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        precisions = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_true != c) & (y_pred == c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            precisions.append(precision)

        return float(np.mean(precisions))


# ============================================================
# Recall Macro
# ============================================================

class RecallMacro(Metric):
    """
    Macro-averaged recall (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        recalls = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fn = np.sum((y_true == c) & (y_pred != c))

            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            recalls.append(recall)

        return float(np.mean(recalls))


# ============================================================
# F1 Macro
# ============================================================

class F1Macro(Metric):
    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        f1_scores = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_pred == c) & (y_true != c))
            fn = np.sum((y_true == c) & (y_pred != c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

            if precision + recall == 0:
                f1 = 0.0
            else:
                f1 = 2 * precision * recall / (precision + recall)

            f1_scores.append(f1)

        return float(np.mean(f1_scores))


# ============================================================
# Metric Collection (Run multiple metrics at once)
# ============================================================

class MetricCollection:
    """
    Utility to compute multiple metrics at once.
    """

    def __init__(self, metrics: dict):
        """
        metrics: dict[name -> Metric instance]
        """
        self.metrics = metrics

    def __call__(self, y_true, y_pred):
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}


# ✅**Final Submission**